# 🧬 pg_bio: In-Database 3D Structure Parsing
In this notebook, we parse a raw `.pdb` file natively in PostgreSQL.

In [ ]:
import pandas as pd
from pgbio import PgBioClient
import psycopg

# Connect to pg_bio
conn_str = "postgresql://localhost:28818/pg_bio"
client = PgBioClient(conn_str)

# Create a small dummy PDB file
pdb_content = """ATOM      1  N   ALA A   1       0.000   0.000   0.000  1.00  0.00           N
ATOM      2  CA  ALA A   1       1.458   0.000   0.000  1.00  0.00           C
ATOM      3  C   ALA A   1       2.009   1.404   0.000  1.00  0.00           C
ATOM      4  O   ALA A   1       3.227   1.583   0.000  1.00  0.00           O
ATOM      5  CB  ALA A   1       2.000  -0.741  -1.196  1.00  0.00           C
"""
with open('/tmp/test_protein.pdb', 'w') as f:
    f.write(pdb_content)


## Native Parsing and Z-Order Indexing

In [ ]:
query = """
    SELECT 
        atom_id, 
        residue, 
        x, y, z,
        residue_z_index(ROW(x, y, z, residue)::ResidueCoord) as morton_z_order
    FROM parse_pdb_file('/tmp/test_protein.pdb');
"""
with client._get_conn() as conn:
    df = pd.read_sql_query(query, conn)
df
